# 0. Environment setting

## Libary import

In [1]:
import pandas as pd
from datetime import date, timedelta
import re
import requests
import json
import urllib3
from difflib import get_close_matches
import ctypes
import threading
import time
import os
from urllib.parse import quote
requests.packages.urllib3.disable_warnings()
from io import StringIO

## Qualtrics Credentials

In [2]:
# ============================================================
# CONFIGURATION 
# ============================================================

API_TOKEN = "dZZueEgbaPrCuSTXEtIp2sz0EqlJNxg93jYEod7U"   # <-- insert your token
DATA_CENTER = "iad1"
SURVEYS_ID = {
    "CMP" : "SV_6zfVwcNb8KrpiPI",
    "RGE" : "SV_4HFHCtt06kaQv0a",
    "NSE" : "SV_eyAxdOqDXvF3qu2"
}

## iQor SFTP Credentials

In [3]:
BASE_URL = 'https://mft.iqor.com'
USERNAME = 'Iberdrola'
PASSWORD = 'KaC#9eta'

## Success/failure message
- Note: This just just work on a windows environment this need to be removed and replaced with an email/team notification

In [4]:
def popup(message, title="Info", timeout=10, is_error=False):
    MB_OK = 0x0
    icon = 0x10 if is_error else 0x40  # ERROR vs INFORMATION
    
    def show_box():
        ctypes.windll.user32.MessageBoxW(0, message, title, MB_OK | icon)
    
    t = threading.Thread(target=show_box)
    t.start()
    t.join(timeout=timeout)
    
    hwnd = ctypes.windll.user32.FindWindowW(None, title)
    if hwnd:
        ctypes.windll.user32.PostMessageW(hwnd, 0x0010, 0, 0)

# Keep your original names as simple wrappers if you want
def popup_info(message, title="Success", timeout=10):
    popup(message, title, timeout, is_error=False)

def popup_error(message, title="Error", timeout=10):
    popup(message, title, timeout, is_error=True)

## Get sharepoint landing folder

In [5]:
def get_onedrive_path():
    return os.path.join(os.path.expanduser("~"), "OneDrive - IBERDROLA S.A")

def get_sharepoint_folder(opco):
    """
    Returns the landing folder path for a given OpCo.
    opco: 'CMP', 'RGE', or 'NSE'
    """
    OPCO_FOLDERS = {
        "CMP": "iQor_CMP",
        "RGE": "iQor_RGE",
        "NSE": "iQor_NYSEG"
    }
    
    if opco not in OPCO_FOLDERS:
        raise ValueError(f"Unknown OpCo: {opco}. Must be one of {list(OPCO_FOLDERS.keys())}")
    
    onedrive_root = get_onedrive_path()
    return os.path.join(
        onedrive_root,
        "iQor-Avangrid - General",
        OPCO_FOLDERS[opco]
    )

def get_output_path(filename, opco):
    """
    Returns (full_path, folder_exists) for a given filename and OpCo.
    """
    sharepoint_folder = get_sharepoint_folder(opco)
    if os.path.exists(sharepoint_folder):
        return os.path.join(sharepoint_folder, filename), True
    return filename, False

# 1. Extract

## Initial data extraction
Task:
- Get all theraw data from the file despite the format
- Get the column names from the file context (If the order changes the dataframe keeps it consistent)

In [6]:
session = requests.Session()
session.verify = False
session.auth = (USERNAME, PASSWORD)
session.get(BASE_URL + '/files', timeout=15)

<Response [200]>

In [7]:
# ============================================================
# EXTRACT
# ============================================================

session = requests.Session()
session.verify = False
session.auth = (USERNAME, PASSWORD)

# Warmup - allow server to fully establish session
session.get(BASE_URL + '/files', timeout=15)
time.sleep(2)
session.get(BASE_URL + '/Report/Survey%20Reports/', timeout=15)
time.sleep(2)

SFTP_FOLDERS = {
    "CMP": "/Report/Survey%20Reports/CMP/",
    "RGE": "/Report/Survey%20Reports/RGE/",
    "NSE": "/Report/Survey%20Reports/NSE/",
}

def list_dir(path):
    r = session.get(BASE_URL + path, timeout=15)
    files, folders = [], []
    for line in r.text.strip().splitlines()[1:]:
        parts = line.split()
        if len(parts) < 9:
            continue
        perms, _, owner, group, size, month, day, time_str, *name_parts = parts
        name = ' '.join(name_parts)
        if name in ('.', '..'):
            continue
        is_dir = perms.startswith('d')
        entry = {
            'name' : name,
            'size' : int(size),
            'date' : f'{month} {day} {time_str}',
            'type' : 'DIR' if is_dir else 'FILE'
        }
        (folders if is_dir else files).append(entry)
    return files, folders

def get_latest_file(opco):
    folder_path  = SFTP_FOLDERS[opco]
    files, _     = list_dir(folder_path)
    time.sleep(1)
    reports      = [
        f for f in files
        if 'Daily Survey Report_' in f['name']
        and 'Triage' not in f['name']
    ]
    if not reports:
        raise FileNotFoundError(f"No Daily Survey Reports found for {opco}")
    reports.sort(key=lambda f: f['name'].split('_')[-1].replace('.csv', ''))
    latest       = reports[-1]
    print(f"[{opco}] Latest file : {latest['name']}")
    print(f"[{opco}] Server date : {latest['date']}")
    encoded_name = quote(latest['name'])
    r            = session.get(BASE_URL + folder_path + encoded_name, timeout=30)
    r.raise_for_status()
    parts        = latest['name'].replace('.csv', '').split('_')
    company      = parts[0].split()[0]
    # ⚠️ TYPE THESE TWO LINES BY HAND IN JUPYTER
    report_date  = pd.to_datetime(parts[-1], format='%Y%m%d')
    df           = pd.read_csv(StringIO(r.text))
    # ⚠️ END
    df['company']     = company
    df['report_date'] = report_date
    print(f"[{opco}] Shape       : {df.shape}")
    return df, latest['name'], r.text

# --- Run ---
dataframes = {}
raw_files  = {}
for opco in SFTP_FOLDERS:
    try:
        df, filename, raw_text = get_latest_file(opco)
        dataframes[opco]       = df
        raw_files[opco]        = (filename, raw_text)
    except Exception as e:
        print(f"[{opco}] ERROR: {e}")

df_cmp = dataframes.get("CMP")
df_rge = dataframes.get("RGE")
df_nse = dataframes.get("NSE")

[CMP] Latest file : CMP Daily Survey Report_20260427.csv
[CMP] Server date : Apr 28 07:00:16
[CMP] Shape       : (177, 17)
[RGE] Latest file : RGE Daily Survey Report_20260427.csv
[RGE] Server date : Apr 28 07:01:47
[RGE] Shape       : (183, 16)
[NSE] Latest file : NSE Daily Survey Report_20260427.csv
[NSE] Server date : Apr 28 07:01:17
[NSE] Shape       : (357, 16)


In [ ]:
print("hello world")

In [ ]:

def extract_data(input_path):

    # Read the full sheet (no assumptions about columns)
    raw = pd.read_excel(input_path, header=None, dtype=str)

    # Row 7 contains the question text
    question_row = raw.iloc[6].fillna("").astype(str)

    # Data starts at row 9
    data = raw.iloc[8:].reset_index(drop=True)

    # Prepare final column names list
    final_cols = []

    for col_idx, col_series in data.items():
        sample_value = col_series.dropna().astype(str).iloc[0] if col_series.dropna().size > 0 else ""
        question_text = question_row[col_idx].lower()

        # -------------------------
        # METADATA COLUMN DETECTION
        # -------------------------

        # ID
        if re.match(r"^[UE]\d+", sample_value):
            final_cols.append("ID")
            continue

        # Name (column immediately right of ID)
        if len(final_cols) > 0 and final_cols[-1] == "ID":
            final_cols.append("Name")
            continue

       # Date/Time (column immediately right of Name)
        if len(final_cols) > 0 and final_cols[-1] == "Name":
            final_cols.append("Date/Time")
            continue

        # InteractionID (alphanumeric)
        if re.match(r"^[A-Za-z0-9]{12,}$", sample_value) and not sample_value.startswith("+"):
            final_cols.append("InteractionID")
            continue

        # Phone Number
        if sample_value.startswith("+"):
            final_cols.append("Phone Number")
            continue

        # Survey Name
        if any(x in sample_value.lower() for x in ["survey", "surv"]):
            final_cols.append("Survey Name")
            continue

        # Work Group
        if any(x in sample_value.lower() for x in ["cc", "vendor", "new"]):
            final_cols.append("Work Group")
            continue

        # -------------------------
        # SCORING COLUMN DETECTION
        # -------------------------

        qt = question_text  # shorthand

        if "recommend" in qt:
            final_cols.append("NPS")
            continue

        if "resolve" in qt or "call back" in qt:
            final_cols.append("FCR")
            continue

        if "help" in qt:
            final_cols.append("E_H")
            continue

        if any(x in qt for x in ["clear", "explain", "explaine"]):
            final_cols.append("C_E")
            continue

        if "satisfied" in qt:
            final_cols.append("CSAT")
            continue

        if any(x in qt for x in ["payment", "billing", "outage"]):
            final_cols.append("Call Reason")
            continue

        # If nothing matches, mark as Unknown
        final_cols.append(f"Unknown_{col_idx}")

    # Apply the detected column names
    data.columns = final_cols

    # Convert Date/Time to proper format
    if "Date/Time" in data.columns:
        data["Date/Time"] = pd.to_datetime(data["Date/Time"], errors="coerce")
        data["Date/Time"] = data["Date/Time"].dt.strftime("%m/%d/%Y %H:%M:%S")
    
    # Convert scoring columns to integers
    score_cols = ["NPS", "FCR", "E_H", "C_E", "CSAT", "Call Reason"]

    for col in score_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce").astype("Int64")

    
    #display(data.head(5))

    return data



# 2. Transform

## Add metadata columns (Survey Status/Completion & Tag)
Tasks:
- Delete empty columns
- Compute Survey Completion and Tag Fields
- Reorder the fields matching the Qualtrics survey order

In [ ]:
def transform_data(df):

    # 0. Remove empty columns
    df = df.dropna(axis=1, how="all")

    # 1. Detect unknown columns
    expected_cols = ["ID", "Name", "Date/Time", "InteractionID", "Phone Number",
                     "Survey Name", "Work Group", "NPS", "FCR", "E_H",
                     "C_E", "CSAT", "Call Reason"]

    unknown_cols = [c for c in df.columns if c not in expected_cols]

    # Windows popup warning
    if unknown_cols:
        import ctypes
        message = f"Unknown columns detected:\n{unknown_cols}"
        ctypes.windll.user32.MessageBoxW(0, message, "ETL Warning", 0x40)

    # 2. Validate scoring columns
    required_cols = ["NPS", "FCR", "E_H", "C_E", "CSAT", "Call Reason"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required scoring columns: {missing}")

    # 3. Validate Work Group and CSAT exist before popping
    if "Work Group" not in df.columns:
        raise ValueError("Column 'Work Group' is missing from the dataset.")

    if "CSAT" not in df.columns:
        raise ValueError("Column 'CSAT' is missing from the dataset.")

    # Move Columns
    csat = df.pop("CSAT")
    work_group = df.pop("Work Group")

    df.insert(3, "Work Group", work_group)
    df.insert(7, "CSAT", csat)

    # Null handling of Name and ID
    df[["ID", "Name"]] = df[["ID", "Name"]].ffill()

    # Create Survey Status/Completion column
    df["Survey Status"] = df[required_cols].notna().all(axis=1)
    df["Survey Status"] = df["Survey Status"].map({True: "Complete", False: "Abandoned"})

    # Tag column
    df["Tag"] = df["Work Group"].str.contains("test", case=True, na=False).map({True: "Test", False: ""})

    # Convert scoring fields into integers
    for col in required_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    #display(df.head(5))
    return df


## Prepare format for Qualtrics
Tasks:
- Get real field names form the qualtrics targeted survey
- Map dataframe field with official survey field names
- Add extra metadata rows for qualtrics understanding

In [ ]:
def prepare_for_qualtrics(df):

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}"

    headers = {
        "X-API-TOKEN": API_TOKEN
    }

    response = requests.get(url, headers=headers, verify=False)
    
    # Convert API response to JSON
    survey_json = response.json()

    # Pull the Qualtrics question metadata
    label_map = survey_json["result"]["questions"]

    # QID → questionName / questionText
    qid_to_name = {qid: q["questionName"] for qid, q in label_map.items()}
    qid_to_text = {qid: q["questionText"] for qid, q in label_map.items()}

    # questionName → QID
    name_to_qid = {name: qid for qid, name in qid_to_name.items()}

    # All official Qualtrics labels (questionName)
    qualtrics_names = list(name_to_qid.keys())

    # 1) Fuzzy‑match df columns to Qualtrics questionName
    mapped_cols = {}
    used_names = set()

    for col in df.columns:
        match = get_close_matches(col, qualtrics_names, n=1, cutoff=0.6)
        if match:
            new_name = match[0]
            if new_name in used_names:
                new_name = col  # avoid duplicates
            mapped_cols[col] = new_name
            used_names.add(new_name)
        else:
            mapped_cols[col] = col  # leave unmapped as‑is

    df = df.rename(columns=mapped_cols)

    # 2) Row 2: questionText (aligned to final column names)
    row2 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        row2.append(qid_to_text.get(qid, "") if qid else "")

    # 3) Row 3: {"ImportId": "QIDx_TEXT"} as STRING
    row3 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        if qid:
            row3.append(f'{{"ImportId": "{qid}_TEXT"}}')
        else:
            row3.append("")

    # 4) Stack as extra rows (no new columns)
    # 4) Stack as extra rows (NO new columns)
    hdr1 = pd.DataFrame([list(df.columns)], columns=df.columns)  # row 1: questionName
    hdr2 = pd.DataFrame([row2], columns=df.columns)              # row 2: questionText
    hdr3 = pd.DataFrame([row3], columns=df.columns)              # row 3: ImportId/QID_TEXT

    df_final = pd.concat([hdr1, hdr2, hdr3, df.reset_index(drop=True)], ignore_index=True)

    # Remove duplicated header row and reset index
    df_final = df_final.iloc[1:].reset_index(drop=True)

   
    display(df_final.head(5))
    return df_final


# 3. Load

## File csv creation after transformation
Tasks:
- Create repository file
- Create Queatrics ready temporary file

In [ ]:
def load_data(df, output_path):
    df.to_csv(output_path, index=False, encoding="utf-8")

    return


## Source folder mapping

In [ ]:
# Extract Date fields
today = date.today()

# If today is Monday (weekday() == 0), use last Saturday
if today.weekday() == 0:
    effective_date = today - timedelta(days=2)
else:
    effective_date = today

# Extract Date fields
source_year = effective_date.year
source_month = effective_date.strftime("%B")
source_day = effective_date.strftime("%d")
source_weekday = effective_date.weekday()
landing_yesterday = effective_date - timedelta(days=1)

#print(source_year, source_month, source_day, source_weekday, landing_yesterday)


## Post to Qualtrics

In [ ]:
def upload_to_qualtrics(file_path, DATA_CENTER, SURVEY_ID, API_TOKEN):
    """
    Uploads a CSV file to Qualtrics using the Import Responses API.
    Expects a fully formatted Qualtrics-ready CSV at file_path.
    """

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}/import-responses"

    headers = {
        "X-API-TOKEN": API_TOKEN,
        "Content-Type": "text/csv",
        "charset": "UTF-8"
    }

    print(f"\nUploading file to Qualtrics: {file_path}")

    with open(file_path, "rb") as f:
        response = requests.post(
            url,
            headers=headers,
            data=f,
            verify=False
        )

    print("\n=== RAW RESPONSE TEXT ===")
    print(response.text)
    print("=========================\n")

    try:
        result = response.json()
        print("Upload response (parsed):")
        print(json.dumps(result, indent=4))
        return result
    except Exception:
        print("Could not parse JSON response.")
        return response.text


# Execute

In [ ]:
try :
    input_file_path = rf"\\clornas01\DIGITAL_COE_CS_DATA\data_delivery\qualtrics\NY_post_call_survey\daily\{source_year}\{source_month}\{source_day}\NY Feedback Daily.xls"
    #input_file_path = r"\\clornas01\DIGITAL_COE_CS_DATA\data_delivery\qualtrics\NY_post_call_survey\daily\2026\April\03\NY Feedback Daily.xls"
    #output_file_path = r"~\Desktop\NY Feedback Daily Cleaned March 5.csv"
    #output_file_path = "NY_qualtrics_upload6.csv"   # local temp file
    #output_file_path = f"NY Feedback Daily {landing_yesterday}.csv"
    filename = f"NY Feedback Daily {landing_yesterday}.csv"
    output_file_path, used_sharepoint = get_output_path(filename)

    temp_file_path = "NY_qualtrics_upload.csv"   # local temp file
    data = extract_data(input_file_path)
    cleaned_data = transform_data(data)
    load_data(cleaned_data, output_file_path)
    ready_data = prepare_for_qualtrics(cleaned_data)
    load_data(ready_data, temp_file_path)
    result = upload_to_qualtrics(temp_file_path, DATA_CENTER, SURVEY_ID, API_TOKEN)
    
except Exception as e:
    error_message = f"Error running the program:\n{e}"

    if today.weekday() == 0:
        error_message += "\n\nMonday run — Friday folder expected."
    elif today.weekday() == 6:
        error_message += "\n\nWeekend run — folder may be empty."

    popup_error(error_message)

else:
    popup_info(filename,"Upload successful")
